In [ ]:
import os
import json
import base64
import requests
import pandas as pd
import time
from pathlib import Path
from PIL import Image
from io import BytesIO
from openai import OpenAI

# --- Configuration ---
OPENROUTER_API_KEY = "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" # Add your actual key here!
MODEL_NAME = "google/gemma-3-27b-it:free"

NGROK_URL = "https://3325-194-27-149-203.ngrok-free.app/v1"
# MODELS_TO_TEST = ["llava:7b", "qwen2.5vl:7b", "qwen3-vl:8b"]
MODELS_TO_TEST = ["llava:7b"]
# MODELS_TO_TEST = ["qwen2.5vl:7b"]
# MODELS_TO_TEST = ["qwen3-vl:8b"]

In [ ]:
# --- 1. Image Encoding ---
def encode_and_resize_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- 2. API Queries ---
def query_vision_model(prompt, base64_image, max_retries=3):
    """Sends the image and prompt to generate the Turtle code, tracking response time."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    strict_prompt = f"""{prompt}

INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution. 
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ]
    }
    
    for attempt in range(max_retries):
        try:
            # Start the timer before sending the request
            start_time = time.time()
            
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120 
            )
            response.raise_for_status() 
            
            # Stop the timer the moment we get the response
            end_time = time.time()
            generation_duration = round(end_time - start_time, 2)
            
            result_text = response.json()['choices'][0]['message']['content']
            
            # Return both the text and the time it took
            return result_text, generation_duration
            
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: Generation Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: Generation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    # If all attempts fail, return an error string and 0.0 seconds
    return "ERROR_MAX_RETRIES_REACHED", 0.0

def evaluate_code_equivalence(model_code, ground_truth_code, max_retries=3):
    """Sends the generated code and truth code back to the API for evaluation."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    eval_prompt = f"""Compare the following two Python Turtle scripts. 
Analyze their logic and determine if they will generate the exact same visual shape/output on the screen.
Answer ONLY with 'Yes' or 'No'. Do not explain your reasoning.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}"""

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": eval_prompt}]
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120 
            )
            response.raise_for_status() 
            raw_response = response.json()['choices'][0]['message']['content'].strip()
            
            if "yes" in raw_response.lower():
                return "Yes"
            elif "no" in raw_response.lower():
                return "No"
            else:
                return raw_response 
                
        except requests.exceptions.Timeout:
            print(f"   ⏳ Eval Attempt {attempt + 1}/{max_retries}: Evaluation Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Eval Attempt {attempt + 1}/{max_retries}: Evaluation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR"

# --- 3. Dataset Loading ---
def load_turtle_bench_from_folders(dataset_path):
    tasks_dir = dataset_path / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ ERROR: Could not find the Tasks folder at {tasks_dir}")
        return None

    data = []
    for task_folder in tasks_dir.iterdir():
        if not task_folder.is_dir():
            continue
            
        task_id = task_folder.name
        text_dir = task_folder / "QA" / "text"
        code_dir = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"
        
        if text_dir.exists():
            for query_file in text_dir.glob("q*.txt"):
                question_id = query_file.stem
                
                with open(query_file, 'r', encoding='utf-8') as f:
                    query_text = f.read().strip()
                
                code_file = code_dir / f"{question_id}_code.txt"
                code_text = "No code found."
                if code_file.exists():
                    with open(code_file, 'r', encoding='utf-8') as f:
                        code_text = f.read().strip()
                
                data.append({
                    'task_id': task_id,
                    'question_id': question_id,
                    'query_text': query_text,
                    'code_text': code_text,
                    'task_folder': str(task_folder),
                    'query_path': str(query_file),
                    'code_path': str(code_file),
                    'image_path': str(image_path)
                })
    return pd.DataFrame(data)

def find_turtlebench_automatically():
    current_dir = Path.cwd()
    for parent in [current_dir, *current_dir.parents]:
        if parent.name.lower() == "thesis":
            target_path = parent / "DTurtleBench"
            if target_path.exists():
                return target_path
    
    fallback_path = Path.home() / "Desktop" / "Thesis" / "DTurtleBench"
    if fallback_path.exists():
        return fallback_path
    return None

# --- 4. Processing Pipeline & Evaluation ---
def process_dataset_to_csv(df):
    print(f"🚀 Starting inference on {len(df)} questions using {MODEL_NAME}...")
    
    safe_model_name = MODEL_NAME.replace("/", "_").replace(":", "_")
    gen_csv = f"TurtleBench_{safe_model_name}_generation.csv"
    eval_csv = f"TurtleBench_{safe_model_name}_evaluation.csv"
    
    processed_tasks = set()
    file_exists_gen = os.path.isfile(gen_csv)
    file_exists_eval = os.path.isfile(eval_csv)
    
    if file_exists_eval:
        try:
            existing_df = pd.read_csv(eval_csv)
            if not existing_df.empty and 'task_id' in existing_df.columns and 'question_id' in existing_df.columns:
                processed_tasks = set(existing_df['task_id'].astype(str) + "_" + existing_df['question_id'].astype(str))
                print(f"🔄 Found existing Eval CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    for index, row in df.iterrows():
        task_key = str(row['task_id']) + "_" + str(row['question_id'])
        if task_key in processed_tasks:
            continue 
            
        print(f"\nProcessing Task: {row['task_id']} | Question: {row['question_id']}...")
        
        # 1. Vision Request
        base64_img = encode_and_resize_image(row['image_path'])
        if not base64_img:
            print("   ⚠️ Skipping due to image error.")
            continue
            
        # Unpack the new response format (response, time)
        model_response, generation_time = query_vision_model(row['query_text'], base64_img)
        ground_truth = row['code_text'].strip()
        
        print(f"   ⏱️ Model generated code in {generation_time} seconds.")
        
        # Save Generation CSV (with new generation_time_seconds column)
        gen_df = pd.DataFrame([{
            'task_id': row['task_id'],
            'question_id': row['question_id'],
            'query_text': row['query_text'],
            'model_response': model_response,
            'ground_truth_code': ground_truth,
            'generation_time_seconds': generation_time
        }])
        gen_df.to_csv(gen_csv, mode='a', header=not file_exists_gen, index=False)
        file_exists_gen = True 
        
        # --- PAUSE AFTER FIRST REQUEST ---
        print("   ⏳ Waiting 10 seconds before sending evaluation request...")
        time.sleep(10)
        
        # 2. Text Evaluation Request
        print("   🔍 Sending codes to model for output equivalence check...")
        same_output_prediction = evaluate_code_equivalence(model_response, ground_truth)
        
        # Save Evaluation CSV (with new generation_time_seconds column)
        eval_df = pd.DataFrame([{
            'task_id': row['task_id'],
            'question_id': row['question_id'],
            'query_text': row['query_text'],
            'model_response': model_response,
            'ground_truth_code': ground_truth,
            'same_output': same_output_prediction,
            'generation_time_seconds': generation_time
        }])
        eval_df.to_csv(eval_csv, mode='a', header=not file_exists_eval, index=False)
        file_exists_eval = True 
        
        processed_tasks.add(task_key)
        eval_mark = "✅" if same_output_prediction == "Yes" else "❌"
        print(f"   {eval_mark} Evaluation completed (Prediction: {same_output_prediction}) | Saved to CSVs.")
        
        # --- PAUSE AFTER SECOND REQUEST ---
        print("   ⏳ Waiting 10 seconds before moving to the next task...")
        time.sleep(10)

# --- Execution ---
if __name__ == "__main__":
    print("Searching for the DTurtleBench dataset...")
    dynamic_dataset_path = find_turtlebench_automatically()

    if dynamic_dataset_path:
        df = load_turtle_bench_from_folders(dynamic_dataset_path)

        if df is not None and not df.empty:
            print(f"✅ Successfully loaded {len(df)} questions.\n")
            print("=" * 60)
            
            # Using .sample(2) to safely test 2 rows first
            process_dataset_to_csv(df.sample(2))
            
            print("\n🎉 Run complete!")
        else:
            print("⚠️ The folders were found, but no qX.txt files could be parsed.")
    else:
        print("❌ Could not find the 'DTurtleBench' folder.")

Searching for the DTurtleBench dataset...
✅ Successfully loaded 260 questions.

🚀 Starting inference on 2 questions using google/gemma-3-27b-it:free...

Processing Task: 49 | Question: q1...
   ⏱️ Model generated code in 5.03 seconds.
   ⏳ Waiting 10 seconds before sending evaluation request...
   🔍 Sending codes to model for output equivalence check...
   ❌ Evaluation completed (Prediction: No) | Saved to CSVs.
   ⏳ Waiting 10 seconds before moving to the next task...

Processing Task: 90 | Question: q1...
❌ Error processing image d:\Users\Desktop\Thesis\DTurtleBench\Tasks\90\image\90.png: [Errno 2] No such file or directory: 'd:\\Users\\Desktop\\Thesis\\DTurtleBench\\Tasks\\90\\image\\90.png'
   ⚠️ Skipping due to image error.

🎉 Run complete!


In [ ]:
# --- 1. Image Encoding ---
def encode_and_resize_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"{base64_str}" # Return raw base64 for the OpenAI client
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- Configuration for Ngrok ---
client = OpenAI(
    base_url=NGROK_URL,
    api_key="ollama",
    timeout=180.0 
)

# --- 2. API Queries ---
def query_vision_model(model_name, prompt, base64_image, max_retries=3):
    """Sends the image and prompt to generate the Turtle code, tracking response time."""
    strict_prompt = f"""{prompt}

INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution. 
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""
    
    for attempt in range(max_retries):
        try:
            start_time = time.time()
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": strict_prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ]
                    }
                ],
                temperature=0.1
            )
            end_time = time.time()
            generation_duration = round(end_time - start_time, 2)
            result_text = response.choices[0].message.content
            return result_text, generation_duration
            
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: Generation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR_MAX_RETRIES_REACHED", 0.0

def evaluate_code_equivalence(model_name, model_code, ground_truth_code, max_retries=3):
    """Sends the generated code and truth code back to the API for evaluation."""
    eval_prompt = f"""Compare the following two Python Turtle scripts. 
Analyze their logic and determine if they will generate the exact same visual shape/output on the screen.
Answer ONLY with 'Yes' or 'No'. Do not explain your reasoning.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": eval_prompt}],
                temperature=0.1
            )
            raw_response = response.choices[0].message.content.strip()
            
            if "yes" in raw_response.lower():
                return "Yes"
            elif "no" in raw_response.lower():
                return "No"
            else:
                return raw_response 
                
        except Exception as e:
            print(f"   ❌ Eval Attempt {attempt + 1}/{max_retries}: Evaluation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR"

# --- 3. Dataset Loading ---
def load_turtle_bench_from_folders(dataset_path):
    tasks_dir = Path(dataset_path) / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ ERROR: Could not find the Tasks folder at {tasks_dir}")
        return None

    data = []
    for task_folder in tasks_dir.iterdir():
        if not task_folder.is_dir():
            continue
            
        task_id = task_folder.name
        text_dir = task_folder / "QA" / "text"
        code_dir = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"
        
        if text_dir.exists():
            for query_file in text_dir.glob("q*.txt"):
                question_id = query_file.stem
                
                with open(query_file, 'r', encoding='utf-8') as f:
                    query_text = f.read().strip()
                
                code_file = code_dir / f"{question_id}_code.txt"
                code_text = "No code found."
                if code_file.exists():
                    with open(code_file, 'r', encoding='utf-8') as f:
                        code_text = f.read().strip()
                
                data.append({
                    'task_id': task_id,
                    'question_id': question_id,
                    'query_text': query_text,
                    'code_text': code_text,
                    'task_folder': str(task_folder),
                    'query_path': str(query_file),
                    'code_path': str(code_file),
                    'image_path': str(image_path)
                })
    return pd.DataFrame(data)

def find_turtlebench_automatically():
    current_dir = Path.cwd()
    for parent in [current_dir, *current_dir.parents]:
        if parent.name.lower() == "thesis":
            target_path = parent / "DTurtleBench"
            if target_path.exists():
                return target_path
    
    fallback_path = Path.home() / "Desktop" / "Thesis" / "DTurtleBench"
    if fallback_path.exists():
        return fallback_path
    return None

# --- 4. Processing Pipeline & Evaluation ---
def process_dataset_to_csv(df):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting inference on {len(df)} questions using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        gen_csv = f"{safe_model_name}_generation.csv"
        eval_csv = f"{safe_model_name}_evaluation.csv"
        
        processed_tasks = set()
        if os.path.isfile(eval_csv):
            try:
                existing_df = pd.read_csv(eval_csv)
                if not existing_df.empty and 'task_id' in existing_df.columns and 'question_id' in existing_df.columns:
                    processed_tasks = set(existing_df['task_id'].astype(str) + "_" + existing_df['question_id'].astype(str))
                    print(f"🔄 Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Resume error: {e}")
                
        for index, row in df.iterrows():
            task_key = str(row['task_id']) + "_" + str(row['question_id'])
            if task_key in processed_tasks:
                continue 
                
            print(f"\nProcessing Task: {row['task_id']} | Question: {row['question_id']}...")
            
            base64_img = encode_and_resize_image(row['image_path'])
            if not base64_img:
                print("   ⚠️ Skipping due to image error.")
                continue
                
            model_response, generation_time = query_vision_model(model_name, row['query_text'], base64_img)
            ground_truth = row['code_text'].strip()
            
            print(f"   ⏱️ Model generated code in {generation_time} seconds.")
            
            # Save Generation CSV
            gen_df = pd.DataFrame([{
                'task_id': row['task_id'],
                'question_id': row['question_id'],
                'query_text': row['query_text'],
                'model_response': model_response,
                'ground_truth_code': ground_truth,
                'generation_time_seconds': generation_time
            }])
            gen_df.to_csv(gen_csv, mode='a', header=not os.path.isfile(gen_csv), index=False)
            
            print("   🔍 Sending codes to model for output equivalence check...")
            same_output_prediction = evaluate_code_equivalence(model_name, model_response, ground_truth)
            
            # Save Evaluation CSV
            eval_df = pd.DataFrame([{
                'task_id': row['task_id'],
                'question_id': row['question_id'],
                'query_text': row['query_text'],
                'model_response': model_response,
                'ground_truth_code': ground_truth,
                'same_output': same_output_prediction,
                'generation_time_seconds': generation_time
            }])
            eval_df.to_csv(eval_csv, mode='a', header=not os.path.isfile(eval_csv), index=False)
            
            processed_tasks.add(task_key)
            eval_mark = "✅" if same_output_prediction == "Yes" else "❌"
            print(f"   {eval_mark} Evaluation completed (Prediction: {same_output_prediction}) | Saved to CSVs.")
            time.sleep(5)

# --- Execution ---
if __name__ == "__main__":
    print("Searching for the DTurtleBench dataset...")
    dynamic_dataset_path = find_turtlebench_automatically()

    if dynamic_dataset_path:
        df = load_turtle_bench_from_folders(dynamic_dataset_path)

        if df is not None and not df.empty:
            print(f"✅ Successfully loaded {len(df)} questions.\n")
            print("=" * 60)
            
            # Test sample
            process_dataset_to_csv(df.head(2))
            
            print("\n🎉 Run complete!")
        else:
            print("⚠️ No qX.txt files could be parsed.")
    else:
        print("❌ Could not find the 'DTurtleBench' folder.")


Searching for the DTurtleBench dataset...
✅ Successfully loaded 260 questions.


🚀 Starting inference on 2 questions using llava:7b...

Processing Task: 61 | Question: q1...


KeyboardInterrupt: 